In [1]:
import pandas as pd
import numpy as np

# 1. 讀取原始資料 (請確保檔名正確)
# 假設你的原始檔案叫做 source_data.xlsx
# 錯誤訊息顯示找不到 '資料來源 1.xlsx' 這個檔案。
# 請確認 '資料來源 1.xlsx' 檔案已經上傳到 Colab 環境中，
# 或者檢查檔案名稱和路徑是否正確。
# 如果檔案尚未上傳，你可以取消下面兩行的註解來上傳檔案：
# from google.colab import files
# uploaded = files.upload()
df = pd.read_excel('資料來源 1.xlsx')

# 2. 基礎資料清理
df['Date'] = pd.to_datetime(df['日期'])  # 轉換為日期格式
df = df.sort_values('Date').reset_index(drop=True) # 建議先由小到大排，處理完再轉回倒序
df['File'] = '資料來源 1' # 建立 File 欄位
df['S0'] = df['收盤價']    # 確保標的價格欄位名稱正確

# 3. 建立 2024-2025 合約到期對照表 (核心邏輯)
expiry_data = {
    'Contract': ['202401', '202402', '202403', '202404', '202405', '202406',
                 '202407', '202408', '202409', '202410', '202411', '202412', '202501'],
    'ContractExpiryDate': pd.to_datetime([
        '2024-01-17', '2024-02-21', '2024-03-20', '2024-04-17', '2024-05-15', '2024-06-19',
        '2024-07-17', '2024-08-21', '2024-09-18', '2024-10-16', '2024-11-20', '2024-12-18', '2025-01-15'
    ])
}
df_expiry = pd.DataFrame(expiry_data)

# 4. 使用 merge_asof 進行「近月合約」對接 (相當於 Excel 的 XLOOKUP 模糊比對)
# direction='backward' 在這裡配上 sorted 資料，代表找「下一個最近的到期日」
df = pd.merge_asof(df, df_expiry, left_on='Date', right_on='ContractExpiryDate', direction='forward')

# 5. 計算 Maturity (存續期間)
df['Maturity'] = (df['ContractExpiryDate'] - df['Date']).dt.days

# 6. 補入 Rf (1-3月為 0.585，其餘為 0.71)
df['Rf'] = df['Date'].dt.month.apply(lambda x: 0.585 if x <= 3 else 0.71)

# 7. 格式調整 (若需要倒序，在此轉回)
df = df.sort_values('Date', ascending=False)

# 8. 輸出最終結果
output_columns = ['Date', 'File', 'S0', 'Contract', 'ContractExpiryDate', 'Maturity', 'Rf']
df_final = df[output_columns]
df_final.to_excel('年度索引檔_學號.xlsx', index=False)

print("資料處理完成！已匯出 Excel 檔。")

FileNotFoundError: [Errno 2] No such file or directory: '資料來源 1.xlsx'